
Forward pass - start with the smallest components that have the least (zero) dependencies and build up documentation as we move toward higher level modules.  The idea here is to look at how things work and deduce what they do.


In [ ]:
import json
from pathlib import Path
from typing import List, Dict, Set, Tuple
import networkx as nx
import os


os.environ.pop("http_proxy", None)
os.environ.pop("https_proxy", None)
os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)

# --- Config ---
PROJECT_ROOT = Path('../..').resolve()  # Adjust as needed to find the project root
GRAPH_PATH = PROJECT_ROOT / "projects/doc_gen/internal/code_graph.gml"
DATABASE_PATH = PROJECT_ROOT / "projects/doc_gen/internal/code_graph.json"
PRIORITY_PATH = PROJECT_ROOT / "projects/doc_gen/internal/forward_pass_schedule.json"
DOX_DIR = PROJECT_ROOT / "projects/doc_gen/internal/generated_dox"
XML_DIR = PROJECT_ROOT / "projects/doc_gen/doxygen_output/xml"

SRC_DIR = PROJECT_ROOT / 'src'
TEMP_DIR = PROJECT_ROOT / 'tmp'

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/qte2333/repos/legacy


In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
import legacy_common.doxygen_parse as dp

db = dp.load_db(DATABASE_PATH)

In [ ]:
import legacy_common.doxygen_graph as dg

# --- Load graph and priority ---
graph: nx.DiGraph = dg.load_graph(GRAPH_PATH)

2025-06-21 13:00:44.896 | INFO     | doxygen_graph:load_graph:457 - Graph loaded from /Users/qte2333/repos/legacy/projects/doc_gen/internal/code_graph.gml, nodes: 14557, edges: 49889


In [ ]:
import legacy_common.doc_db as doc_db

In [6]:
# for the first forward pass, we only care about functions because we are trying to
# deduce what things do by their implementation.  After the backward pass we can
# do a second forward refinement pass that summarizes how things intended to be used.

# add nodes we don't want to document to the visited set
skip = set(
    n for n,d in graph.nodes(data=True)
#    if d.get("type") in (dg.EntityType.BODY, dg.EntityType.DECLARATION)
    if d.get('type') not in (
        dg.EntityType.COMPOUND, dg.EntityType.MEMBER
    ) or d.get('kind') not in (
        'function',
    )
    # ) or d.get('kind') in (
    #     'dir',
    #     'file',
    #     'namespace',
    #     'friend',
    #     'enumvalue',
    # )
)

priority_order = dg.create_visit_list(graph, skip_fn=lambda n: n in skip)

with PRIORITY_PATH.open('w', encoding='utf-8') as f:
    json.dump(priority_order, f, indent=2)

# `priority_order` now contains a prioritized list of documentation targets

Skipped 12382, Scheduled     5 nodes with fan-in 0. Total visited: 12387/14557
Skipped     1, Scheduled     0 nodes with fan-in 0. Total visited: 12388/14557
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 12389/14557
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 12390/14557
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 12391/14557
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 12392/14557
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 12393/14557
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 12394/14557
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 12395/14557
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 12396/14557
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 12397/14557
Skipped     0, Scheduled     1 nodes with fan-in 1. Total visited: 12398/14557
Skipped     0, Scheduled     1 nodes with fan-in 1. 

In [7]:
# clear all responses in the database
if False:
    for cid, docs in doc_db.docs.items():
        for sig, doc in docs.items():
            if 'response' in doc and not isinstance(doc['response'], dict):
                del doc['response']
        doc_db.save_docs(cid)

In [ ]:
from legacy_common.llm_utils import call_openai
from doc_gen.gen_common import format_dependency_summaries, document_entity

OPENAI_MODEL = "gpt-4.1-nano"
OPENAI_MAX_TOKENS = None

# --- Prompt template ---
def create_prompt(code, docs, kind, dep_summaries):
    prompt = f"You are generating documentation for a C/C++ project. The goal is to produce a clean, structured Doxygen-style comment block."

    if len(dep_summaries) > 0:
        prompt += f"\nThe following entities are used in this implementation:\n{'\n'.join(dep_summaries)}"

    prompt += f"\n\nDocument the following {kind}:\n\n"
    # if len(docs) > 0:
    #     prompt += f"{docs}\n"
    prompt += code

    prompt += f"\nRespond ONLY with a JSON object with the following fields:"

    example = """\n\nExample:\n{\n  "brief": "...",\n  "details": "...\""""

    if kind in ('class', 'struct', 'file', 'namespace'):
        prompt += f"\n- brief: a one-line summary of what state and/or behavior the {kind} encapsulates"
        prompt += f"\n- details: a more detailed description of the {kind} that fully describes its usage"
    if kind in ('function',):
        prompt += f"\n- brief: a one-line summary of what the {kind} does"
        prompt += f"\n- details: a more detailed description of the {kind}'s usage and effects, with sufficient detail for treating the {kind} as a black box"
        prompt += f"\n- params: a dictionary mapping parameter names to their usage in this {kind}"
        prompt += f"\n- return: a description of the return value and what it represents"
        prompt += f"\n- tparams: (for templates) a dictionary mapping template parameter names to usage in this {kind}"
        prompt += f"\n- throws: a description of the exceptions that may be thrown and the conditions they represent (use 'null' if not applicable)"
        example += """,\n  "params": {{"param1": "...", "param2": "..."}},\n  "return": "..." """
    if kind in ('variable', 'enum', 'typedef', 'define', 'friend'):
        prompt += f"\n- brief: a one-line summary of what the {kind} represents"
        prompt += f"\n- details: a more detailed description of the {kind}'s role and usage"

    example += "\n}"
    prompt += example
    return prompt

skip_completed = True
dry_run = False

# Count nodes in the graph whose "kind" is in the specified list
skip_kinds = ('variable', 'typedef', 'define', 'namespace', 'file')

nodes_to_document: List[Tuple[str, dp.DoxygenEntity]] = []
for item in priority_order:
    node_id, kind = item['id'], item['kind']

    if kind in skip_kinds:
        continue

    eid = dg.get_body_eid(db, node_id)
    entity = db.get(eid)

    doc = doc_db.get_doc(entity.id.compound, entity.signature)

    if skip_completed and doc_db.is_doc_complete(doc):
        if doc.state == doc_db.DocumentState.EXTRACTED_SUMMARY:
            doc.state = doc_db.DocumentState.GENERATED_SUMMARY

        continue

    nodes_to_document.append((node_id, entity))

completed = 0

# --- Forward pass cell ---
for node_id, entity in nodes_to_document:

    completed += 1
    print(f"{completed:4}/{len(nodes_to_document):4}: {str(entity.id)} - {entity.signature}")

    code_data = dg.get_code_body(PROJECT_ROOT, db, graph, node_id)
    if not code_data:
        continue

    body = code_data['code']
#    print(f"{entity_name}: {file} - {start}:{end}")

    if not body:
        print("stopping on nonexisting code")
        break


    deps = list(format_dependency_summaries(db, graph, node_id))

    # temporary: break until we can test dependency summary inclusion
    # if len(deps) > 0:
    #     print("halting to test dependency summary inclusion")
    #     break

    prompt = create_prompt(body, None, entity.kind, list(deps))

    if dry_run:
        doc_json = "(dry run)"
    else:
        response_text = call_openai(prompt, OPENAI_MODEL, OPENAI_MAX_TOKENS)
        try:
            response_text = response_text.lstrip("```json").rstrip("```")
            doc_json = json.loads(response_text)
        except Exception as e:
            raise Exception(f"Failed to parse model output as JSON: {e}")

        doc = doc_db.get_doc(entity.id.compound, entity.signature)
        if not doc:
            doc = document_entity(entity)

        doc.state = doc_db.DocumentState.GENERATED_SUMMARY
        doc.prompt = prompt
        doc.response = doc_db.DoxygenFields(**doc_json)
        doc_db.save_docs(entity.id.compound)
    
    # print(doc_json)
    print(prompt)
    # break

print("[✓] Forward pass complete.")

[✓] Forward pass complete.


In [9]:
# commit changes to docs
def commit_docs():
    for cid, cid_docs in doc_db.docs.items():
        for sig, doc in cid_docs.items():
            if not doc.response:
                continue

            print(f"updating doc {cid} {sig}")
            doc.brief = doc.response.brief or doc.brief
            doc.details = doc.response.details or doc.details
            if doc.response.params:
                doc.params = doc.params or {}
                doc.params.update(doc.response.params)
            if doc.response.tparams:
                doc.tparams = doc.tparams or {}
                doc.tparams.update(doc.response.tparams)
            doc.returns = doc.response.returns or doc.returns
            doc.throws = doc.response.throws or doc.throws

            doc.response = None
            doc.prompt = None

        doc_db.save_docs(cid)

commit_docs()

In [10]:
sigs = {}

for cid, data in graph.nodes(data=True):
    if data['type'] != dg.EntityType.COMPOUND:
        continue

    compound = db.compounds[dp.EntityID(compound=cid)]
    sigs[cid] = set([compound.signature])

    for mid in graph.predecessors(cid):
        if graph.edges[mid, cid]['type'] != dg.Relationship.CONTAINED_BY:
            continue

        member = db.get(dp.EntityID(compound=cid, member=mid))
        sigs.setdefault(cid, set()).add(member.signature)

for cid, signatures in sigs.items():
    doc_db.docs[cid] = {
        k:v for k,v in doc_db.docs[cid].items()
        if k in signatures
    }
    doc_db.save_docs(cid)